# Mycomaterial patent extraction — consolidation and restructuring

**Input:** `../mycodemo/data/interim/germini/` (or auto-detected `data/interim/germini/` inside the project).  
**Output:** `data/processed/patent_processes/` within `mycodemo`.

This notebook:
1. Loads all `*_extracted.csv` files; inventories are kept separately.
2. Preserves original values, units, citations, and filenames.
3. Assigns stable process and operation IDs.
4. Excludes records explicitly identified as **aerial mycelium cultivation**; uncertain records are flagged, not silently discarded.
5. Creates linked **processes**, **operations**, **parameters**, and **material properties** CSVs.
6. Produces product-review candidates for **acoustic boards, bricks, leather, and foam**. Product tags are *keyword-based suggestions*, not verified intended uses.

**Caution:** Patent claims are not necessarily experimental results. No patent values are independently verified here.


In [ ]:
from pathlib import Path
import pandas as pd
import re
import hashlib
from IPython.display import display

# Works when notebook is run from mycodemo/, mycodemo/notebooks/, or an adjacent folder.
candidates = [
    Path.cwd() / "data/interim/germini",
    Path.cwd().parent / "data/interim/germini",
    Path.cwd() / "mycodemo/data/interim/germini",
    Path.cwd().parent / "mycodemo/data/interim/germini",
    Path.cwd() / "../mycodemo/data/interim/germini",
]
INPUT_DIR = next((p.resolve() for p in candidates if p.is_dir()), None)
if INPUT_DIR is None:
    raise FileNotFoundError("Cannot find data/interim/germini. Set INPUT_DIR manually to your actual folder.")
PROJECT_ROOT = INPUT_DIR.parents[2]  # mycodemo/data/interim/germini -> mycodemo
OUTPUT_DIR = PROJECT_ROOT / "data/processed/patent_processes"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Input:", INPUT_DIR, "\nOutput:", OUTPUT_DIR)


In [ ]:
detail_files = sorted(INPUT_DIR.glob("*_extracted.csv"))
inventory_files = sorted(INPUT_DIR.glob("*inventory*.csv"))
if not detail_files:
    raise FileNotFoundError(f"No *_extracted.csv files found in {INPUT_DIR}")
frames = []
required = ["Patent","Process/Example","Step Number","Stage","Operation",
            "Parameter","Value","Unit","Evidence Type","Source Citation"]
for file in detail_files:
    df = pd.read_csv(file, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    df.columns = df.columns.str.strip()
    missing = set(required) - set(df.columns)
    if missing:
        print(f"SKIP {file.name}: missing {sorted(missing)}")
        continue
    df = df[required].copy()
    df["source_file"] = file.name
    df["source_row"] = range(2, len(df) + 2)  # CSV header occupies row 1
    frames.append(df)
if not frames:
    raise ValueError("No valid detailed extraction files.")
raw = pd.concat(frames, ignore_index=True)
raw.to_csv(OUTPUT_DIR / "all_extracted_original.csv", index=False)
print("Loaded:", len(raw), "parameter rows from", len(frames), "files")
display(raw.groupby("source_file").size().rename("rows").reset_index())


## Scope filtering

Only explicitly aerial-growing records are excluded. A patent mentioning forced aeration is **not automatically** aerial mycelium cultivation; solid-state substrate composites can also use forced aeration. Ambiguous cases are exported for review. The original data remain available in `all_extracted_original.csv`.


In [ ]:
def norm(s):
    return re.sub(r"\s+", " ", str(s).strip()).casefold()

def stable_id(prefix, *parts):
    token = "||".join(norm(x) for x in parts)
    return prefix + hashlib.sha1(token.encode("utf-8")).hexdigest()[:12]

raw["process_id"] = [stable_id("P_", p, ex) for p, ex in zip(raw["Patent"], raw["Process/Example"])]
# One operation is a process + step number + stage + operation description
raw["operation_id"] = [
    stable_id("O_", pid, step, stage, op)
    for pid, step, stage, op in zip(raw.process_id, raw["Step Number"], raw.Stage, raw.Operation)
]
# Conservative process-level scope classification; inspect the review file.
process_text = (raw.groupby("process_id")[["Patent","Process/Example"]]
                .first().reset_index())
def scope_status(text):
    s = norm(text)
    if any(x in s for x in ["aerial mycelium", "aerial hyph", "airmycelium", "void-space mycelium"]):
        return "exclude_aerial"
    if any(x in s for x in ["forced-aeration", "forced aeration", "airflow", "aerated"]):
        return "review_aeration_not_necessarily_aerial"
    return "include"
process_text["scope_status"] = process_text["Process/Example"].map(scope_status)
raw = raw.merge(process_text[["process_id","scope_status"]], on="process_id", how="left")
process_text.to_csv(OUTPUT_DIR / "scope_review.csv", index=False)
included = raw.loc[raw.scope_status != "exclude_aerial"].copy()
excluded = raw.loc[raw.scope_status == "exclude_aerial"].copy()
excluded.to_csv(OUTPUT_DIR / "excluded_aerial_rows.csv", index=False)
print("Included rows:", len(included), "| Explicitly excluded aerial rows:", len(excluded))
display(process_text["scope_status"].value_counts())


## Product categories — suggestions only

A single process can be relevant to several products. These tags are derived from wording in the *process name* and, where available, the original inventory's intended product field. They are **not** proof of material suitability or measured performance. Untagged processes remain in the database.


In [ ]:
inventory = pd.DataFrame()
if inventory_files:
    # Prefer revised inventory when available
    selected = sorted(inventory_files, key=lambda p: ("_2" in p.stem, p.name))[-1]
    inventory = pd.read_csv(selected, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    inventory["source_file"] = selected.name
    inventory.to_csv(OUTPUT_DIR / "inventory_reference.csv", index=False)
    print("Inventory reference:", selected.name, "| rows:", len(inventory))

patterns = {
    "acoustic_board": r"acoustic|sound absor|soundproof|wall panel|acoustic panel",
    "brick_block": r"brick|block|masonry|construction block|building block",
    "leather": r"leather|hide|textile|flexible sheet|tanning",
    "foam": r"foam|foamed|cellular foam|lightweight cushion",
}
processes = (included.groupby("process_id", as_index=False)
             .agg(patent=("Patent","first"), process_example=("Process/Example","first"),
                  scope_status=("scope_status","first"),
                  parameter_rows=("Parameter","size"),
                  operations_count=("operation_id","nunique"),
                  source_file=("source_file","first")))
for product, pattern in patterns.items():
    processes[product] = processes.process_example.str.contains(pattern, case=False, regex=True, na=False)
processes["product_tag_status"] = "suggested_from_process_name_only"
processes["needs_product_review"] = True
processes.to_csv(OUTPUT_DIR / "processes.csv", index=False)
display(processes[["patent","process_example",*patterns.keys()]].head(20))


In [ ]:
operations = (included[["operation_id","process_id","Step Number","Stage","Operation"]]
              .drop_duplicates().rename(columns={
                  "Step Number":"step_number", "Stage":"stage", "Operation":"operation"
              }).sort_values(["process_id","step_number","operation"]))
operations.to_csv(OUTPUT_DIR / "operations.csv", index=False)

# Preserve every original row, including NR and descriptive (non-numeric) values.
parameters = included[[
    "process_id","operation_id","Parameter","Value","Unit",
    "Evidence Type","Source Citation","source_file","source_row"
]].rename(columns={
    "Parameter":"parameter", "Value":"value_original", "Unit":"unit_original",
    "Evidence Type":"evidence_type", "Source Citation":"source_citation"
}).copy()
parameters.insert(0, "parameter_id", [
    stable_id("R_", f, row) for f, row in zip(parameters.source_file, parameters.source_row)
])
parameters.to_csv(OUTPUT_DIR / "parameters.csv", index=False)

# Conservative property detection: property rows are a *subset* of parameters.
# Flag candidates rather than assuming all are measured or comparable.
property_regex = (r"tensile|compressive|compression strength|flexural|elastic modulus|"
                  r"thermal conductivity|sound absorption|acoustic absorption|"
                  r"water absorption|density|elongation|tear strength|abrasion|porosity")
material_properties = parameters.loc[
    parameters.parameter.str.contains(property_regex, case=False, regex=True, na=False)
].copy()
material_properties["measurement_verified"] = False
material_properties["note"] = "Candidate property; verify whether measured, claimed, or target"
material_properties.to_csv(OUTPUT_DIR / "material_properties_candidates.csv", index=False)

# Many-to-many process-to-product review table; includes untagged processes.
product_review = processes[["process_id","patent","process_example",*patterns.keys()]].melt(
    id_vars=["process_id","patent","process_example"],
    var_name="product_category", value_name="keyword_match"
)
product_review["confirmed_relevance"] = ""  # manually fill: direct / potential / not_relevant
product_review["review_notes"] = ""
product_review.to_csv(OUTPUT_DIR / "product_classification_review.csv", index=False)
print("Processes:", len(processes), "Operations:", len(operations),
      "Parameters:", len(parameters), "Property candidates:", len(material_properties))


In [ ]:
# Basic integrity checks
assert parameters.process_id.isin(processes.process_id).all()
assert parameters.operation_id.isin(operations.operation_id).all()
assert operations.process_id.isin(processes.process_id).all()
assert parameters.parameter_id.is_unique
assert len(parameters) + len(excluded) == len(raw)
summary = pd.DataFrame({
    "table": ["processes","operations","parameters","material_properties_candidates",
              "product_classification_review","scope_review"],
    "rows": [len(processes),len(operations),len(parameters),len(material_properties),
             len(product_review),len(process_text)]
})
display(summary)
print("Saved CSV files to:", OUTPUT_DIR)
print("NEXT: Review scope_review.csv and product_classification_review.csv before interpreting product comparisons.")
